# Práctica 8 — Modelo Víctima
**Seguridad y Privacidad · UCM**

Dataset: **CIFAR-10** | Arquitectura: **Mini-ResNet** | Framework: **PyTorch**

Este notebook entrena y guarda el modelo víctima que otra pareja intentará robar.

In [ ]:
!pip install torch torchvision matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Dataset — CIFAR-10

**Por qué hace difícil el robo:** CIFAR-10 tiene imágenes en color 32×32 con 10 clases muy distintas entre sí. La variabilidad intra-clase es alta (un "coche" puede ser rojo, azul, visto de lado o de frente), así que el atacante necesita muchas consultas para cubrir bien el espacio de entrada.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# Data augmentation fuerte en entrenamiento
# Hace que el modelo aprenda features invariantes a la posición y el color
# → el atacante no puede replicar esa invarianza simplemente consultando imágenes limpias
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

CLASSES = ['avión','automóvil','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']
print(f'Train: {len(train_set)} muestras | Test: {len(test_set)} muestras')

In [ ]:
# Visualizar ejemplos del training set (sin normalizar, para que se vean bien)
raw_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=False,
                                        transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    img, label = raw_set[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(CLASSES[label], fontsize=7)
    ax.axis('off')
plt.suptitle('Ejemplos CIFAR-10', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## 2. Arquitectura — Mini-ResNet

Tres bloques residuales con stride progresivo. Las **skip connections** obligan al modelo a aprender transformaciones residuales no lineales que son muy difíciles de aproximar desde el exterior con un modelo más simple.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return self.relu(out)


class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock(64,  64,  stride=1)
        self.layer2 = ResidualBlock(64,  128, stride=2)  # 32→16
        self.layer3 = ResidualBlock(128, 256, stride=2)  # 16→8
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.drop   = nn.Dropout(0.4)
        self.fc     = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


model = MiniResNet(num_classes=10).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parámetros totales: {total_params:,}')

## 3. Entrenamiento

- **Weight decay:** mejor generalización, fronteras de decisión más consistentes.
- **LR scheduling (cosine annealing):** convergencia a un mínimo más estrecho → las fronteras de decisión son más complejas.

In [ ]:
EPOCHS    = 50
LR        = 0.1
MOMENTUM  = 0.9
WD        = 5e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WD, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    criterion_eval = nn.CrossEntropyLoss()
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion_eval(logits, labels).item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(model, test_loader)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['test_loss'].append(te_loss)
    history['test_acc'].append(te_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
              f'test_loss={te_loss:.4f}  test_acc={te_acc:.4f}')

## 4. Evaluación

In [ ]:
# Curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'],  label='Test')
ax1.set_title('Loss'); ax1.set_xlabel('Época'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'],  label='Test')
ax2.set_title('Accuracy'); ax2.set_xlabel('Época'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Mini-ResNet CIFAR-10')
plt.tight_layout()
plt.show()

final_test_loss, final_test_acc = evaluate(model, test_loader)
print(f'\nAccuracy final en test: {final_test_acc:.4f}  ({final_test_acc*100:.2f}%)')

In [ ]:
# Accuracy por clase
class_correct = torch.zeros(10)
class_total   = torch.zeros(10)

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for c in range(10):
            mask = labels == c
            class_correct[c] += (preds[mask] == c).sum().item()
            class_total[c]   += mask.sum().item()

class_acc = class_correct / class_total

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(CLASSES, class_acc.numpy(), color='steelblue', edgecolor='white')
ax.axhline(final_test_acc, linestyle='--', color='red', label=f'Media ({final_test_acc:.3f})')
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy por clase — test set')
ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplos correctos e incorrectos
raw_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=False,
                                         transform=transforms.ToTensor())

model.eval()
correct_imgs, wrong_imgs = [], []

with torch.no_grad():
    for imgs_norm, labels in test_loader:
        preds = model(imgs_norm.to(DEVICE)).argmax(1).cpu()
        for i in range(len(labels)):
            entry = (imgs_norm[i], labels[i].item(), preds[i].item())
            if preds[i] == labels[i] and len(correct_imgs) < 5:
                correct_imgs.append(entry)
            elif preds[i] != labels[i] and len(wrong_imgs) < 5:
                wrong_imgs.append(entry)
        if len(correct_imgs) >= 5 and len(wrong_imgs) >= 5:
            break

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, (img_t, true, pred) in enumerate(correct_imgs):
    img = img_t.permute(1, 2, 0).numpy()
    img = (img * np.array(CIFAR10_STD) + np.array(CIFAR10_MEAN)).clip(0, 1)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'✓ {CLASSES[pred]}', fontsize=8, color='green')
    axes[0, i].axis('off')

for i, (img_t, true, pred) in enumerate(wrong_imgs):
    img = img_t.permute(1, 2, 0).numpy()
    img = (img * np.array(CIFAR10_STD) + np.array(CIFAR10_MEAN)).clip(0, 1)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'✗ pred:{CLASSES[pred]}\nreal:{CLASSES[true]}', fontsize=7, color='red')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correctas', fontsize=9)
axes[1, 0].set_ylabel('Erróneas',  fontsize=9)
plt.suptitle('Predicciones del modelo víctima')
plt.tight_layout()
plt.show()

## 5. Guardar el modelo

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'MiniResNet',
    'dataset': 'CIFAR-10',
    'test_accuracy': final_test_acc,
    'normalization': {'mean': CIFAR10_MEAN, 'std': CIFAR10_STD},
}, 'victim_model.pth')

print('Modelo guardado en victim_model.pth')
print(f'Accuracy final: {final_test_acc*100:.2f}%')